# Week 6: Policy Gradient Methods

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview 

This lab demonstrates what Sutton and Barto call the one-step actor-critic method, an instance of policy gradient methods described in Reinforcement Learning, Chapter 13.

Policy gradient methods are a better approach to Lunar Lander than state-action value based methods for a couple of reasons. Sutton and Barto write, "With continuous policy parameterization the action probabilities change smoothly as a function of the learned parameter, whereas in ε-greedy selection the action probabilities may change dramatically for an arbitrarily small change in the estimated action values, if that change results in a different action having the maximal value" (p. 324). Lunar Lander benefits from smoother, more nuanced actions. Additionally, with a continuous state and action space, function approximation would be necessary for a value-based approach, and this would likewise suffer from instability in action selection.

The demonstration uses Gymnasium's LunarLanderContinuous-v3 environment. LunarLanderContinuous-v2 has been deprecated in favor of v3, which is functionally equivalent for our purposes. The environment has a continuous observation space in 8 dimensions and a continuous action space in 2 dimensions. The environment provides dense, shaped rewards at every step. The agent is rewarded for proximity to the landing pad, low velocity, and a level orientation, while fuel consumption penalizes unnecessary engine use (0.03 points per frame for side engines, 0.3 for the main engine). Each leg making contact with the ground adds 10 points. A terminal reward of +100 or -100 is issued for a safe landing or crash respectively. An episode scoring 200 points or more is considered solved. An episode terminates under three conditions: the lander crashes, it drifts outside the bounding box, or it comes to rest without colliding.

The two-network architecture divides responsibilities cleanly: the critic learns to estimate state values using semi-gradient TD methods, producing an advantage signal at each step. The actor uses that signal to adjust action probabilities, increasing the likelihood of actions that exceeded expectations and decreasing those that fell short. Because the actor outputs a Gaussian policy parameterized by mean μ(s) and log standard deviation log σ(s), policy entropy is a direct function of σ. Higher σ means more spread, more exploration, and higher entropy. Tracking entropy over training therefore serves as a window into how confidently the policy is committing to its actions.

We take a temporal-differencing approach with bootstrapping rather than a Monte Carlo approach, which is appropriate for this problem because each step carries substantial information from which to update policy estimates, without waiting for the episode to conclude.

Given the dense, multi-dimensional reward signal, agents are expected to learn relatively quickly. Average return per episode across seeds should climb steadily and approach the solution threshold of 200 points within the first few hundred episodes. Policy entropy should exhibit an inverse relationship with average return: as agents converge on effective action values, the Gaussian standard deviation will narrow, reflecting growing confidence in the learned policy.

## Section 2: Deliverables

GitHub Repository: https://github.com/craig-rudman/MSDS684/tree/main/W6

### Implementation Summary

We implemented a one-step Actor-Critic agent trained on the Gymnasium LunarLanderContinuous-v3 environment. The actor outputs a Gaussian policy parameterized by mean μ(s) and log standard deviation log σ(s); the critic estimates state values V(s). Both networks share the same architecture: two hidden layers of 64 units each with ReLU activation. Independent agents were trained for 1,000 episodes across 30 random seeds to improve robustness. Final policy evaluation uses ensemble voting, averaging the mean actions μ(s) across all 30 trained actors to reduce variance from any individual seed. Key hyperparameters include an actor learning rate of 1e-4, a critic learning rate of 1e-3, and a discount factor of γ=0.99.

#### Key Results and Analysis

##### Introduction

When we examine training outcomes across 30 independent seeds, average returns climb through the first 200 episodes before plateauing well below the solve threshold of 200 points. A histogram of per-seed performance reveals a bimodal distribution: some seeds failed to learn an effective policy entirely, while most partially learned but plateaued well short of 200. Entropy curves show that agents grew increasingly confident over training, but on average in suboptimal policies. Ensemble voting across all 30 actors suggests that agents tended to learn descent control but not lateral movement, with the area of least engine effort approaching but not precisely centered on the landing pad.

##### Learning Curve: Return per Episode

<img src="../img/return_per_episode.png">

*Figure 1. Mean episode return across 30 independent seeds (solid line) with 95% confidence interval (shaded) over 1,000 episodes per seed. The dashed line marks the solve threshold of 200 points.*

As shown in Figure 1, average returns across 30 seeds rose steadily through approximately the first 200 episodes, with the 95% confidence interval narrowing as seeds converged toward similar early trajectories. After episode 200, the lower bound of the CI leveled off near -300 while the upper bound continued climbing, reflecting the bimodal outcome distribution seen in the seed performance histogram (Figure 2), which suggests that a subset of seeds continued improving while others stalled or failed. The wide CI throughout training likely reflects both between-seed variance driven by initialization sensitivity and the variability of per-episode returns for each seed run.

<img src="../img/seed_performance_distribution.png">

*Figure 2. Distribution of per-seed mean return averaged over the final 100 episodes across 30 seeds. Vertical lines mark the cross-seed mean (24.8) and median (27.5). The dashed line marks the solve threshold of 200 points.*

To illustrate what learning looked like for a successful seed, Figure 3 shows the learning trajectory of the best-performing seed by final 100-episode mean return.

<img src="../img/best_seed_return.png">

*Figure 3. Episode return for the best-performing seed (ms_s11) over 1,000 episodes (light line) with a 50-episode rolling mean overlay (solid line). The dashed line marks the solve threshold of 200 points.*

The rolling mean climbs gradually across all 1,000 episodes, crossing the solve threshold intermittently but never sustaining above 200. Raw episode returns remain highly variable throughout, reflecting the stochastic nature of both the environment and the Gaussian policy.

##### Critic Learning: TD Error

<img src="../img/td_error.png">

*Figure 4. Mean absolute TD error across 30 independent seeds (solid line) with 95% confidence interval (shaded) over 1,000 episodes per seed. TD error is averaged across all steps within each episode before aggregating across seeds.*

Mean absolute TD error declines through approximately the first 250 episodes, suggesting that critics improve their value estimates early in training. After episode 250, the mean plateaus near zero, but periodic spikes in the CI suggest critics may be overfitting to familiar states, producing systematically inaccurate value estimates when encountering unfamiliar regions of the state space. When those estimates are unreliable, the advantage signal degrades, propagating noise to actor updates and contributing to the return variance observed in Figures 1 and 3.

##### Policy Confidence: Entropy over Training

<img src="../img/policy_entropy.png">

*Figure 5. Mean policy entropy across 30 independent seeds (solid line) with 95% confidence interval (shaded) over 1,000 episodes per seed. Entropy is computed from the Gaussian policy's standard deviation σ at each step and averaged within each episode.*

Policy entropy declines steadily across all 1,000 episodes, falling from approximately 2.8 to near zero by the end of training. Unlike the return and TD error curves, the 95% CI is notably narrow, indicating that entropy reduction is consistent across seeds regardless of outcome. This is not the inverse relationship with return predicted in Section 1. Rather, it suggests that the learning mechanism reliably commits actors to whatever policy they have found, whether effective or not.

##### Learned Policy: Ensemble Rollout Trajectories

<img src="../img/learned_policy_rollouts.png">

*Figure 6. Deterministic rollout trajectories from the ensemble policy across three evaluation seeds, using mean actions with no sampling. Each trajectory begins near the top of the frame and terminates when the episode ends. The gold marker is the landing pad at the origin.*

Ensemble rollout trajectories across three evaluation seeds show similar behavioral patterns: controlled descent with initial lateral drift away from the landing pad, followed by late corrections that bend trajectories toward the target. Episodes terminate near but not on the landing pad, consistent with the suboptimal returns observed across the seed distribution. The ensemble policy demonstrates partial competence in both axes of control, but insufficient precision for reliable landing.

##### Learned Policy: Actor Mean Action Heatmap

<img src="../img/actor_mean_action.png">

*Figure 7. The L2 norm of the ensembled mean action vector (i.e. main engine thrust and lateral thrust) across all 30 trained actors, plotted over a grid of x-position and altitude with all other state dimensions held fixed at zero. Color intensity indicates total action magnitude; the gold marker is the landing pad at the origin.*

The heatmap reveals a gradient in action magnitude with bands converging on a valley that descends toward the landing pad, suggesting the ensemble learned both descent and some degree of lateral control. However, the asymmetry of the gradient relative to the target indicates this control was incomplete. The distortion is consistent with policies overfit to observed states rather than generalizing across the full state space, and with the lateral drift observed in the rollout trajectories.

##### Conclusion

Training produced evidence of learning: returns improved steadily through the first 200 episodes and the ensemble demonstrated partial descent and lateral control. However, returns were noisy and never sustained above the solve threshold, and the heatmap confirms that lateral control was not reliably learned. One-step TD bootstrapping may have given critics too narrow a view, particularly early in training when episodes were short and crash-heavy. N-step or Monte Carlo returns would condition updates on fuller episode outcomes, and entropy regularization could slow premature policy collapse before lateral control has time to develop.

## Section 3: AI Use Reflection

### 3.1 Initial Interaction

I shared the Lab 6 assignment and asked the AI to help design an Actor-Critic implementation for LunarLanderContinuous-v2. My initial prompt established a working agreement: design-first, object-oriented, and test-driven, with development driven by a TODO backlog in the notebook. The AI asked clarifying questions about architecture, class boundaries, and metrics storage, then helped construct the backlog. No code was produced.

### 3.2 Iteration Cycle

For our first iteration, we built the Actor network. Initial code had forward returning a torch.distributions.Normal object. I prompted: "The forward pass should return mean μ(s) and log-std log σ(s)." Claude updated the contract and rewrote the test suite. A follow-up code review caught a hidden_sizes slicing bug; I asked Claude to fix it and it refactored the trunk into a cleaner _mlp_body helper.

For our second iteration, we built the Critic. A code review revealed a missing gradient flow test. I prompted: "Add a test confirming grad_fn is not None." Claude added it. Switching from Tanh to ReLU then exposed a brittle test using extreme input magnitudes; I asked Claude to fix it and it rewrote the test using torch.randn observations.

For our third iteration, we built the Agent. A code review caught that next_value was not detached before computing the TD target. I prompted: "Fix the gradient graph so the bootstrap target is treated as fixed." Claude introduced next_value.detach() and rewrote a test that was not actually verifying the done condition numerically.

Subsequent iterations continued to follow this pattern: design review, implementation, code review, fix and adjust.

### 3.3 Critical Evaluation

We caught one significant implementation error: omitting detach() on the bootstrap value target, which would have coupled the actor and critic gradients and prevented convergence. We also deliberated on class responsibilities, ultimately delegating action sampling and entropy computation to the Agent rather than the Actor. Learning was verified by monitoring policy entropy reduction and episode return growth over training.

### 3.4 Learning Reflection

Debugging suggested that the critics were overfitting to frequently observed states, producing unreliable advantage estimates when encountering unfamiliar regions of the state space. This insight emerged through diagnostic conversation with Claude rather than from any single metric. Working iteratively, we connected TD error spikes, entropy collapse, and heatmap distortion into a coherent picture. Next time, I would do more exploratory work on fewer seeds and episodes to try to identify the root causes of bias and overfitting.

## Section 4: Speaker Notes

- **Problem:** Train a one-step Actor-Critic agent on LunarLanderContinuous-v3, a continuous control task requiring both descent and lateral precision to land reliably.
- **Method:** Gaussian policy actor and TD critic, trained independently across 30 seeds for 1,000 episodes each; final evaluation uses ensemble voting across all 30 trained actors.
- **Design decision:** Evaluating all agents after an equal 1,000 episodes and averaging mean actions across all 30 trained actors at inference reduced the influence of any single seed's outcome on the final policy assessment.
- **Key result:** Most seeds partially learned but plateaued well below the solve threshold; the ensemble demonstrated controlled descent and some lateral correction but not reliable landing.
- **Insight:** Entropy collapsed consistently into suboptimal policies across all seeds, suggesting the learning mechanism commits actors to whatever policy they find, effective or not.
- **Challenge:** One-step Actor-Critic updates from immediate rewards rather than episode returns, so the terminal signal from hitting the landing pad carries little weight. Monte Carlo or n-step methods would amplify that signal and may produce better-targeted policies.